# Vitamin OCR → R00~R11 판정 파이프라인

새 사례의 OCR 추출 JSON 경로 하나를 지정하면 판정 결과를 생성합니다. 합성데이터 전용 코드가 아닙니다.


## 1. 데이터 모델


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from datetime import date, datetime
from decimal import Decimal
from enum import StrEnum
from typing import Any


class ReviewState(StrEnum):
    PENDING = "PENDING"
    CONFIRMED = "CONFIRMED"
    CORRECTED = "CORRECTED"


class RuleStatus(StrEnum):
    PASS = "PASS"
    MISMATCH = "MISMATCH"
    REVIEW = "REVIEW"
    REVIEW_HIGH = "REVIEW_HIGH"
    NOT_CHECKABLE = "NOT_CHECKABLE"


@dataclass(slots=True)
class OcrValue:
    value: Any = None
    confidence: float | None = None
    raw_text: str | None = None
    review_state: ReviewState = ReviewState.PENDING

    @property
    def confirmed(self) -> bool:
        return self.review_state in {ReviewState.CONFIRMED, ReviewState.CORRECTED}


@dataclass(slots=True)
class DocumentData:
    document_type: str
    fields: dict[str, OcrValue] = field(default_factory=dict)
    rows: list[dict[str, OcrValue]] = field(default_factory=list)
    source_id: str | None = None

    def value(self, name: str, default: Any = None) -> Any:
        item = self.fields.get(name)
        return default if item is None or item.value is None else item.value


@dataclass(slots=True)
class CaseBundle:
    contract: DocumentData
    timesheet: DocumentData
    payslip: DocumentData
    bank: DocumentData
    case_id: str | None = None


@dataclass(slots=True)
class RuleResult:
    rule_id: str
    status: RuleStatus
    reason: str
    comparisons: dict[str, Any] = field(default_factory=dict)


@dataclass(slots=True)
class EvaluationReport:
    case_id: str | None
    derived: dict[str, Any]
    rules: list[RuleResult]
    review_required: list[dict[str, Any]] = field(default_factory=list)

    def to_dict(self) -> dict[str, Any]:
        def convert(value: Any) -> Any:
            if isinstance(value, (date, datetime)):
                return value.isoformat()
            if isinstance(value, StrEnum):
                return str(value)
            if isinstance(value, Decimal):
                return int(value) if value == value.to_integral_value() else float(value)
            if isinstance(value, dict):
                return {k: convert(v) for k, v in value.items()}
            if isinstance(value, (list, tuple)):
                return [convert(v) for v in value]
            return value

        return convert(asdict(self))


## 2. 값 정규화


In [ ]:
from __future__ import annotations

import re
import unicodedata
from datetime import date, datetime, time
from decimal import Decimal, InvalidOperation
from typing import Any


def normalize_name(value: Any) -> str | None:
    if value is None:
        return None
    text = unicodedata.normalize("NFKC", str(value)).upper().strip()
    return re.sub(r"[^0-9A-Z가-힣]", "", text) or None


def number(value: Any) -> Decimal | None:
    if value in (None, ""):
        return None
    if isinstance(value, bool):
        return None
    try:
        return Decimal(str(value).replace(",", "").replace("원", "").strip())
    except (InvalidOperation, ValueError):
        return None


def integer(value: Any) -> int | None:
    parsed = number(value)
    return None if parsed is None else int(parsed)


def parse_date(value: Any) -> date | None:
    if isinstance(value, datetime):
        return value.date()
    if isinstance(value, date):
        return value
    if not value:
        return None
    text = str(value).strip().replace(".", "-").replace("/", "-")
    try:
        return date.fromisoformat(text[:10])
    except ValueError:
        return None


def parse_datetime(value: Any) -> datetime | None:
    if isinstance(value, datetime):
        return value
    parsed_date = parse_date(value)
    if parsed_date and len(str(value).strip()) <= 10:
        return datetime.combine(parsed_date, time.min)
    try:
        return datetime.fromisoformat(str(value).strip().replace("Z", "+00:00"))
    except (TypeError, ValueError):
        return None


def parse_time(value: Any) -> time | None:
    if isinstance(value, time):
        return value
    if not value:
        return None
    text = str(value).strip()
    for fmt in ("%H:%M", "%H:%M:%S", "%I:%M %p"):
        try:
            return datetime.strptime(text, fmt).time()
        except ValueError:
            pass
    return None


def normalize_wage_type(value: Any) -> str | None:
    if not value:
        return None
    key = str(value).strip().lower()
    aliases = {
        "시급": "hourly", "시간급": "hourly", "hourly": "hourly",
        "일급": "daily", "daily": "daily",
        "주급": "weekly", "weekly": "weekly",
        "월급": "monthly", "monthly": "monthly",
    }
    return aliases.get(key)


def bool_value(value: Any) -> bool | None:
    if isinstance(value, bool):
        return value
    if value is None:
        return None
    key = str(value).strip().lower()
    if key in {"true", "yes", "y", "1", "예", "유", "제공", "포함"}:
        return True
    if key in {"false", "no", "n", "0", "아니오", "무", "미제공", "미포함"}:
        return False
    return None


## 3. OCR JSON 변환


In [ ]:
from __future__ import annotations

from typing import Any, Protocol



class OcrAdapter(Protocol):
    """문서 파일을 표준 필드로 변환하는 OCR 어댑터 계약."""

    def extract(self, source: str, document_type: str) -> DocumentData: ...


class JsonOcrAdapter:
    """OCR 팀 parser가 만든 JSON을 파이프라인 입력으로 바꾸는 기본 어댑터."""

    def from_dict(
        self,
        payload: dict[str, Any],
        document_type: str,
        *,
        row_key: str | None = None,
        canonical: bool = False,
    ) -> DocumentData:
        metadata = payload.get("_ocr", {})
        confidences = metadata.get("confidence", {})
        raw_text = metadata.get("raw_text", {})
        confirmed = set(metadata.get("confirmed_fields", []))
        corrected = set(metadata.get("corrected_fields", []))

        row_keys = {"rows", "lines", "transactions"}
        fields: dict[str, OcrValue] = {}
        for name, value in payload.get("fields", payload).items():
            if name.startswith("_") or name in row_keys:
                continue
            state = ReviewState.CONFIRMED if canonical else ReviewState.PENDING
            if name in confirmed:
                state = ReviewState.CONFIRMED
            if name in corrected:
                state = ReviewState.CORRECTED
            fields[name] = OcrValue(value, confidences.get(name), raw_text.get(name), state)

        rows = []
        source_rows = payload.get(row_key or "rows", payload.get("rows", []))
        for row in source_rows:
            converted = {
                name: OcrValue(value, review_state=ReviewState.CONFIRMED)
                for name, value in row.items()
            }
            # 합성데이터에는 사용자 선택 UI가 없으므로 포함된 거래를 확정 거래로 취급한다.
            rows.append(converted)
        return DocumentData(document_type, fields, rows, payload.get("_source_id"))


def apply_user_review(document: DocumentData, corrections: dict[str, Any]) -> None:
    """UI에서 확정하거나 수정한 값을 canonical 값으로 반영한다."""
    for name, value in corrections.items():
        if name not in document.fields:
            document.fields[name] = OcrValue(value=value, review_state=ReviewState.CORRECTED)
            continue
        item = document.fields[name]
        if value == item.value:
            item.review_state = ReviewState.CONFIRMED
        else:
            item.value = value
            item.review_state = ReviewState.CORRECTED


## 4. R00~R11 룰 엔진


In [ ]:
from __future__ import annotations

import calendar
from datetime import date, datetime, timedelta
from decimal import Decimal
from typing import Any, Callable


ZERO = Decimal("0")


class RuleEngine:
    """피처정의서 v3.2의 재사용 흐름을 따르는 R00~R11 엔진."""

    DEFAULT_PARAMETERS = {
        "param_wage_tolerance": Decimal("1"),
        "param_fixed_allowance_tolerance": Decimal("10"),
        "param_rate_allowance_base_tolerance": Decimal("10"),
        "param_rate_allowance_hour_tolerance": Decimal("3"),
        "param_minimum_hourly_wage": Decimal("10320"),
        "param_timesheet_min_coverage": Decimal("0.9"),
        "param_hours_tolerance": Decimal("8"),
    }

    def __init__(self, parameters: dict[str, Any] | None = None) -> None:
        self.parameters = dict(self.DEFAULT_PARAMETERS)
        if parameters:
            self.parameters.update(parameters)
        self.derived: dict[str, Any] = {}
        self.case: CaseBundle | None = None

    def evaluate(self, case: CaseBundle) -> tuple[dict[str, Any], list[RuleResult]]:
        self.case = case
        self.derived = {}
        rules: list[Callable[[], RuleResult]] = [
            self.r00, self.r01, self.r02, self.r03, self.r04, self.r05,
            self.r06, self.r07, self.r08, self.r09, self.r10, self.r11,
        ]
        return self.derived, [rule() for rule in rules]

    @property
    def c(self): return self.case.contract  # type: ignore[union-attr]
    @property
    def t(self): return self.case.timesheet  # type: ignore[union-attr]
    @property
    def p(self): return self.case.payslip  # type: ignore[union-attr]
    @property
    def b(self): return self.case.bank  # type: ignore[union-attr]

    def result(self, rule: str, status: RuleStatus, reason: str, **cmp: Any) -> RuleResult:
        return RuleResult(rule, status, reason, cmp)

    def r00(self) -> RuleResult:
        names = [normalize_name(self.c.value("ct_employee_name")),
                 normalize_name(self.t.value("ts_employee_name")),
                 normalize_name(self.p.value("ps_employee_name"))]
        present_names = [v for v in names if v]
        period_start = parse_date(self.p.value("ps_pay_period_start"))
        period_end = parse_date(self.p.value("ps_pay_period_end"))
        work_dates = [parse_date(row.get("ts_work_date").value) for row in self.t.rows if row.get("ts_work_date")]
        work_dates = [v for v in work_dates if v]
        name_match = len(present_names) >= 2 and len(set(present_names)) == 1
        period_match = bool(period_start and period_end and work_dates and all(period_start <= d <= period_end for d in work_dates))
        self.derived.update(cmp_worker_match=name_match, cmp_period_match=period_match)
        if len(present_names) < 2 or not period_start or not period_end:
            return self.result("R00", RuleStatus.NOT_CHECKABLE, "근로자명 또는 산정기간이 부족합니다.")
        if not name_match:
            return self.result("R00", RuleStatus.REVIEW, "문서 간 근로자명이 일치하지 않아 사용자 확인이 필요합니다.", names=present_names)
        if work_dates and not period_match:
            return self.result("R00", RuleStatus.REVIEW, "근무일 일부가 급여 산정기간 밖에 있습니다.")
        self.derived["sys_case_id"] = self.case.case_id
        return self.result("R00", RuleStatus.PASS, "근로자와 산정기간이 연결됩니다.")

    def _contract_hourly(self) -> Decimal | None:
        amount = number(self.c.value("ct_wage_amount"))
        wage_type = normalize_wage_type(self.c.value("ct_wage_type"))
        if amount is None or wage_type is None:
            return None
        if wage_type == "hourly": return amount
        if wage_type == "monthly":
            hours = number(self.c.value("ct_monthly_work_hours"))
            return amount / hours if hours else None
        start, end = parse_time(self.c.value("ct_work_start_time")), parse_time(self.c.value("ct_work_end_time"))
        if wage_type == "daily" and start and end:
            duration = Decimal(str((datetime.combine(date.min, end) - datetime.combine(date.min, start)).seconds / 3600))
            breaks = (number(self.c.value("ct_break_hours")) or ZERO) + (number(self.c.value("ct_break_minutes")) or ZERO) / 60
            return amount / (duration - breaks) if duration > breaks else None
        if wage_type == "weekly":
            hours = number(self.c.value("ct_monthly_work_hours"))
            return amount / (hours / Decimal("4.345")) if hours else None
        return None

    def r01(self) -> RuleResult:
        contract_hourly = self._contract_hourly()
        payslip_hourly = number(self.p.value("ps_ordinary_hourly_wage"))
        if payslip_hourly is None:
            base = number(self.p.value("ps_base_pay"))
            hours = number(self.p.value("ps_paid_regular_hours"))
            payslip_hourly = base / hours if base is not None and hours else None
        self.derived.update(calc_contract_hourly_wage=contract_hourly, calc_payslip_hourly_wage=payslip_hourly)
        ct_type, ps_type = normalize_wage_type(self.c.value("ct_wage_type")), normalize_wage_type(self.p.value("ps_wage_type"))
        if contract_hourly is None or payslip_hourly is None:
            return self.result("R01", RuleStatus.NOT_CHECKABLE, "임금 또는 환산에 필요한 시간이 부족합니다.")
        gap = payslip_hourly - contract_hourly
        tolerance = ZERO if ct_type == ps_type else self.parameters["param_wage_tolerance"]
        self.derived.update(cmp_wage_type_match=ct_type == ps_type, cmp_hourly_wage_gap=gap)
        if abs(gap) > tolerance:
            return self.result("R01", RuleStatus.MISMATCH, "계약 환산시급과 명세서 적용시급이 다릅니다.", gap=gap, tolerance=tolerance)
        return self.result("R01", RuleStatus.PASS, "계약 임금과 명세서 적용단가가 허용범위 내에서 일치합니다.", gap=gap)

    def _timesheet_hours(self) -> tuple[Decimal, Decimal, int]:
        total, overtime, days = ZERO, ZERO, 0
        daily: list[dict[str, Any]] = []
        for row in self.t.rows:
            start = parse_time(row.get("ts_work_start_time").value if row.get("ts_work_start_time") else None)
            end = parse_time(row.get("ts_work_end_time").value if row.get("ts_work_end_time") else None)
            if not start or not end: continue
            raw = Decimal(str((datetime.combine(date.min, end) - datetime.combine(date.min, start)).seconds / 3600))
            breaks = number(row.get("ts_break_minutes").value if row.get("ts_break_minutes") else None)
            if breaks is None: breaks = number(self.c.value("ct_break_minutes")) or ZERO
            hours = max(ZERO, raw - breaks / 60)
            total += hours; overtime += max(ZERO, hours - 8); days += 1
            daily.append({"date": parse_date(row.get("ts_work_date").value) if row.get("ts_work_date") else None, "hours": hours})
        self.derived["ts_daily_actual_hours"] = daily
        self.derived["ts_period_actual_hours"] = total
        self.derived["ts_monthly_actual_hours"] = total
        return total, overtime, days

    def r02(self) -> RuleResult:
        if bool_value(self.t.value("ts_record_available", True)) is False or not self.t.rows:
            return self.result("R02", RuleStatus.NOT_CHECKABLE, "근무기록이 없습니다.")
        actual, overtime, days = self._timesheet_hours()
        expected_days = number(self.p.value("ps_paid_work_days"))
        if expected_days and Decimal(days) / expected_days < self.parameters["param_timesheet_min_coverage"]:
            return self.result("R02", RuleStatus.NOT_CHECKABLE, "근무기록 커버리지가 90% 미만입니다.", coverage=Decimal(days) / expected_days)
        paid = sum((number(self.p.value(k)) or ZERO) for k in ("ps_paid_regular_hours", "ps_paid_overtime_hours", "ps_paid_holiday_hours"))
        paid_overtime = number(self.p.value("ps_paid_overtime_hours")) or ZERO
        total_tol = min(max(Decimal(days) * Decimal("0.083"), Decimal("0.5")), paid * Decimal("0.03")) if paid else Decimal("0.5")
        overtime_tol = max(Decimal("0.5"), Decimal(sum(1 for d in self.derived["ts_daily_actual_hours"] if d["hours"] > 8)) * Decimal("0.083"))
        total_gap, overtime_gap = actual - paid, overtime - paid_overtime
        self.derived.update(cmp_actual_hours_gap=total_gap, cmp_overtime_hours_gap=overtime_gap)
        if abs(total_gap) > total_tol or abs(overtime_gap) > overtime_tol:
            return self.result("R02", RuleStatus.MISMATCH, "근무기록과 명세서 계산시간이 허용범위를 벗어납니다.", actual_gap=total_gap, overtime_gap=overtime_gap)
        return self.result("R02", RuleStatus.PASS, "근무시간이 허용범위 내에서 일치합니다.")

    def r03(self) -> RuleResult:
        checks = []
        fixed_tol = self.parameters["param_fixed_allowance_tolerance"]
        for contract_field, payslip_field, label in (("ct_bonus_amount", "ps_bonus_amount", "상여금"), ("ct_extra_pay_amount", "ps_other_allowance", "기타수당")):
            expected, actual = number(self.c.value(contract_field)), number(self.p.value(payslip_field))
            if expected is not None:
                checks.append((label, actual or ZERO, expected, fixed_tol))
        overtime_hours = number(self.p.value("ps_paid_overtime_hours")) or ZERO
        overtime_rate = number(self.c.value("ct_overtime_hourly_pay"))
        if overtime_rate is None and self.derived.get("calc_contract_hourly_wage") is not None:
            overtime_rate = self.derived["calc_contract_hourly_wage"] * Decimal("1.5")
        if overtime_hours and overtime_rate is not None:
            checks.append(("연장수당", number(self.p.value("ps_overtime_pay")) or ZERO, overtime_rate * overtime_hours,
                           self.parameters["param_rate_allowance_base_tolerance"] + self.parameters["param_rate_allowance_hour_tolerance"] * overtime_hours))
        if not checks:
            return self.result("R03", RuleStatus.NOT_CHECKABLE, "비교할 수당 조건 또는 금액이 없습니다.")
        gaps = {label: actual - expected for label, actual, expected, _ in checks}
        self.derived.update(cmp_bonus_gap=gaps.get("상여금"), cmp_extra_pay_gap=gaps.get("기타수당"), cmp_overtime_pay_gap=gaps.get("연장수당"))
        failed = [(label, gaps[label], tol) for label, _, _, tol in checks if abs(gaps[label]) > tol]
        if failed:
            return self.result("R03", RuleStatus.MISMATCH, "계약상 수당과 명세서 지급액이 다릅니다.", failed=failed)
        return self.result("R03", RuleStatus.PASS, "수당·상여금이 허용범위 내에서 일치합니다.")

    def r04(self) -> RuleResult:
        gaps = {}
        for kind in ("housing", "meal"):
            provided = bool_value(self.c.value(f"ct_{kind}_provided"))
            allowed = number(self.c.value(f"ct_{kind}_cost")) or ZERO
            actual = number(self.p.value(f"ps_{kind}_deduction")) or ZERO
            if provided is False: allowed = ZERO
            gaps[kind] = actual - allowed
        self.derived.update(cmp_housing_deduction_gap=gaps["housing"], cmp_meal_deduction_gap=gaps["meal"])
        if any(v != ZERO for v in gaps.values()):
            return self.result("R04", RuleStatus.MISMATCH, "계약상 숙식비 부담액과 명세서 공제액이 다릅니다.", **gaps)
        return self.result("R04", RuleStatus.PASS, "숙박비·식비 공제가 계약과 일치합니다.")

    def r05(self) -> RuleResult:
        fields = ("ps_base_pay", "ps_bonus_amount", "ps_other_allowance", "ps_weekly_allowance", "ps_overtime_pay", "ps_night_work_pay", "ps_holiday_work_pay")
        values = [number(self.p.value(k)) for k in fields]
        if all(v is None for v in values): return self.result("R05", RuleStatus.NOT_CHECKABLE, "지급항목을 추출하지 못했습니다.")
        calc, printed = sum((v or ZERO for v in values), ZERO), number(self.p.value("ps_gross_pay"))
        self.derived["calc_gross_pay"] = calc
        if printed is None: return self.result("R05", RuleStatus.NOT_CHECKABLE, "명세서 지급액 계가 없습니다.")
        gap = calc - printed; self.derived["cmp_gross_pay_gap"] = gap
        return self.result("R05", RuleStatus.PASS if gap == 0 else RuleStatus.MISMATCH,
                           "지급항목 합계가 일치합니다." if gap == 0 else "지급항목 합계와 인쇄된 지급액 계가 다릅니다.", gap=gap)

    def r06(self) -> RuleResult:
        fields = ("ps_national_pension", "ps_health_insurance", "ps_long_term_care_insurance", "ps_employment_insurance", "ps_income_tax", "ps_local_income_tax", "ps_housing_deduction", "ps_meal_deduction", "ps_other_deduction")
        values = [number(self.p.value(k)) for k in fields]
        if all(v is None for v in values): return self.result("R06", RuleStatus.NOT_CHECKABLE, "공제항목을 추출하지 못했습니다.")
        calc, printed = sum((v or ZERO for v in values), ZERO), number(self.p.value("ps_total_deduction"))
        self.derived["calc_total_deduction"] = calc
        if printed is None: return self.result("R06", RuleStatus.NOT_CHECKABLE, "명세서 공제액 계가 없습니다.")
        gap = calc - printed; self.derived["cmp_total_deduction_gap"] = gap
        return self.result("R06", RuleStatus.PASS if gap == 0 else RuleStatus.MISMATCH,
                           "공제항목 합계가 일치합니다." if gap == 0 else "공제항목 합계와 인쇄된 공제액 계가 다릅니다.", gap=gap)

    def r07(self) -> RuleResult:
        gross, deduction = self.derived.get("calc_gross_pay"), self.derived.get("calc_total_deduction")
        if gross is None or deduction is None: return self.result("R07", RuleStatus.NOT_CHECKABLE, "R05 또는 R06 계산값이 없습니다.")
        calc, printed = gross - deduction, number(self.p.value("ps_net_pay"))
        self.derived["calc_net_pay"] = calc
        if printed is None: return self.result("R07", RuleStatus.REVIEW, "실수령액 필드가 없어 재계산값을 사용합니다.", calculated=calc)
        gap = calc - printed; self.derived["cmp_net_pay_gap"] = gap
        return self.result("R07", RuleStatus.PASS if gap == 0 else RuleStatus.MISMATCH,
                           "실수령액이 일치합니다." if gap == 0 else "재계산 실수령액과 인쇄값이 다릅니다.", gap=gap)

    def _selected_transactions(self) -> list[dict[str, Any]]:
        selected = []
        for row in self.b.rows:
            flag = bool_value(row.get("user_selected_salary_transaction").value if row.get("user_selected_salary_transaction") else None)
            if flag:
                selected.append({k: v.value for k, v in row.items()})
        return selected

    def r08(self) -> RuleResult:
        net = self.derived.get("calc_net_pay")
        selected = self._selected_transactions()
        if net is None or not selected:
            return self.result("R08", RuleStatus.NOT_CHECKABLE, "재계산 실수령액 또는 사용자가 확정한 급여 거래가 없습니다.")
        total = sum((number(row.get("bk_deposit_amount")) or ZERO for row in selected), ZERO)
        gap = total - net
        self.derived.update(calc_salary_deposit_total=total, cmp_deposit_gap=gap, selected_salary_transactions=selected)
        return self.result("R08", RuleStatus.PASS if gap == 0 else RuleStatus.MISMATCH,
                           "선택한 급여 입금합계가 실수령액과 일치합니다." if gap == 0 else "선택한 급여 입금합계와 실수령액이 다릅니다.", gap=gap)

    def _scheduled_payment_date(self) -> date | None:
        period_end = parse_date(self.p.value("ps_pay_period_end"))
        day = number(self.c.value("ct_pay_day"))
        timing = str(self.c.value("ct_pay_timing", "당월"))
        if not period_end or day is None: return None
        year, month = period_end.year, period_end.month
        if "익" in timing or "next" in timing.lower():
            month += 1
            if month == 13: year, month = year + 1, 1
        return date(year, month, min(int(day), calendar.monthrange(year, month)[1]))

    def r09(self) -> RuleResult:
        net, selected = self.derived.get("calc_net_pay"), self.derived.get("selected_salary_transactions", [])
        scheduled = self._scheduled_payment_date()
        if net is None or not selected or scheduled is None:
            return self.result("R09", RuleStatus.NOT_CHECKABLE, "지급일 계산에 필요한 계약조건 또는 선택 거래가 없습니다.")
        cumulative, completion = ZERO, None
        ordered = sorted(selected, key=lambda r: parse_datetime(r.get("bk_transaction_datetime")) or datetime.max)
        for row in ordered:
            cumulative += number(row.get("bk_deposit_amount")) or ZERO
            if cumulative >= net:
                dt = parse_datetime(row.get("bk_transaction_datetime")); completion = dt.date() if dt else None; break
        self.derived.update(calc_scheduled_payment_date=scheduled, calc_actual_payment_completion_date=completion)
        if completion is None: return self.result("R09", RuleStatus.NOT_CHECKABLE, "입금 누적액이 실수령액에 도달하지 않았습니다.")
        if completion != scheduled:
            return self.result("R09", RuleStatus.MISMATCH, "전액 지급 완료일이 계약상 지급일과 다릅니다.", scheduled=scheduled, actual=completion)
        return self.result("R09", RuleStatus.PASS, "전액 지급 완료일이 계약상 지급일과 일치합니다.")

    def r10(self) -> RuleResult:
        actual = self.derived.get("ts_monthly_actual_hours")
        contract = number(self.c.value("ct_monthly_work_hours"))
        daily = self.derived.get("ts_daily_actual_hours", [])
        max_daily = number(self.c.value("ct_max_daily_work_hours"))
        if actual is None or contract is None: return self.result("R10", RuleStatus.NOT_CHECKABLE, "계약시간 또는 근무기록 합계가 없습니다.")
        gap = actual - contract; self.derived["cmp_contract_actual_hours_gap"] = gap
        exceeded = [row for row in daily if max_daily is not None and row["hours"] > max_daily]
        if gap != ZERO or exceeded:
            return self.result("R10", RuleStatus.REVIEW, "계약시간과 실제시간 차이 또는 일일 상한 초과가 있어 사유 확인이 필요합니다.", gap=gap, exceeded_days=exceeded)
        return self.result("R10", RuleStatus.PASS, "계약시간과 실제시간이 일치합니다.")

    # Final v3.2 implementations. These override the earlier compact versions
    # and include every identity/period/wage-type condition in the confirmed spec.
    def r00(self) -> RuleResult:
        names = [normalize_name(self.c.value("ct_employee_name")),
                 normalize_name(self.t.value("ts_employee_name")),
                 normalize_name(self.p.value("ps_employee_name")),
                 normalize_name(self.b.value("bk_account_holder"))]
        period_start = parse_date(self.p.value("ps_pay_period_start"))
        period_end = parse_date(self.p.value("ps_pay_period_end"))
        work_dates = [parse_date(row.get("ts_work_date").value)
                      for row in self.t.rows if row.get("ts_work_date")]
        work_dates = [value for value in work_dates if value]
        name_match = all(names) and len(set(names)) == 1
        employer_match = (
            normalize_name(self.c.value("ct_employer_name"))
            == normalize_name(self.p.value("ps_employer_name"))
        )
        new_selected = bool_value(self.c.value("ct_new_or_reentry_selected")) is True
        change_selected = bool_value(self.c.value("ct_workplace_change_selected")) is True
        selected_start = selected_end = None
        if new_selected != change_selected:
            prefix = "ct_new_or_reentry" if new_selected else "ct_workplace_change"
            selected_start = parse_date(self.c.value(f"{prefix}_start_date"))
            selected_end = parse_date(self.c.value(f"{prefix}_end_date"))
        contract_period_match = bool(
            period_start and period_end and selected_start and selected_end
            and selected_start <= period_start <= period_end <= selected_end
        )
        work_period_match = bool(
            period_start and period_end and work_dates
            and all(period_start <= value <= period_end for value in work_dates)
        )
        self.derived.update(cmp_worker_match=name_match,
                            cmp_employer_match=employer_match,
                            cmp_contract_period_match=contract_period_match,
                            cmp_work_period_match=work_period_match,
                            cmp_period_match=contract_period_match and work_period_match)
        if not all(names) or not period_start or not period_end:
            return self.result("R00", RuleStatus.NOT_CHECKABLE,
                               "근로자명·예금주 또는 산정기간 정보가 부족합니다.")
        if new_selected == change_selected:
            return self.result("R00", RuleStatus.NOT_CHECKABLE,
                               "적용할 계약유형을 하나로 확정할 수 없습니다.")
        if not name_match:
            return self.result("R00", RuleStatus.REVIEW,
                               "문서 간 근로자명 또는 예금주가 일치하지 않습니다.", names=names)
        if not employer_match:
            return self.result("R00", RuleStatus.REVIEW,
                               "계약서와 임금명세서의 사업장이 일치하지 않습니다.")
        if not contract_period_match:
            return self.result("R00", RuleStatus.REVIEW,
                               "급여 산정기간이 선택된 계약기간 밖에 있습니다.")
        if work_dates and not work_period_match:
            return self.result("R00", RuleStatus.REVIEW,
                               "근무일이 급여 산정기간 밖에 있습니다.")
        self.derived["sys_case_id"] = self.case.case_id
        return self.result("R00", RuleStatus.PASS, "근로자와 산정기간이 연결됩니다.")

    def r01(self) -> RuleResult:
        contract_hourly = self._contract_hourly()
        payslip_hourly = number(self.p.value("ps_ordinary_hourly_wage"))
        if payslip_hourly is None:
            base = number(self.p.value("ps_base_pay"))
            hours = number(self.p.value("ps_paid_regular_hours"))
            payslip_hourly = base / hours if base is not None and hours else None
        ct_type = normalize_wage_type(self.c.value("ct_wage_type"))
        ps_type = normalize_wage_type(self.p.value("ps_wage_type"))
        self.derived.update(calc_contract_hourly_wage=contract_hourly,
                            calc_payslip_hourly_wage=payslip_hourly,
                            cmp_wage_type_match=ct_type == ps_type)
        if contract_hourly is None or payslip_hourly is None:
            return self.result("R01", RuleStatus.NOT_CHECKABLE,
                               "임금 또는 환산에 필요한 시간이 부족합니다.")
        gap = payslip_hourly - contract_hourly
        self.derived["cmp_hourly_wage_gap"] = gap
        period_start = parse_date(self.p.value("ps_pay_period_start"))
        period_end = parse_date(self.p.value("ps_pay_period_end"))
        partial_period = bool(
            ct_type == "monthly" and period_start and period_end
            and (period_start.day != 1
                 or period_end.day != calendar.monthrange(period_end.year, period_end.month)[1])
        )
        self.derived["cond_partial_period"] = partial_period
        if partial_period:
            return self.result("R01", RuleStatus.REVIEW,
                               "월급제 일부기간 정산이므로 환산값 확인이 필요합니다.", gap=gap)
        if ct_type != ps_type:
            return self.result("R01", RuleStatus.MISMATCH,
                               "환산금액과 별개로 계약서와 명세서의 임금형태가 다릅니다.", gap=gap)
        if abs(gap) > self.parameters["param_wage_tolerance"]:
            return self.result("R01", RuleStatus.MISMATCH,
                               "계약 환산시급과 명세서 적용시급이 다릅니다.", gap=gap)
        return self.result("R01", RuleStatus.PASS,
                           "계약 임금과 명세서 적용단가가 허용범위 내에서 일치합니다.", gap=gap)

    def _timesheet_hours(self) -> tuple[Decimal, Decimal, int]:
        total, overtime, days = ZERO, ZERO, 0
        daily: list[dict[str, Any]] = []
        period_start = parse_date(self.p.value("ps_pay_period_start"))
        period_end = parse_date(self.p.value("ps_pay_period_end"))
        for row in self.t.rows:
            work_date = parse_date(row.get("ts_work_date").value if row.get("ts_work_date") else None)
            if period_start and period_end and work_date and not (period_start <= work_date <= period_end):
                continue
            start = parse_time(row.get("ts_work_start_time").value if row.get("ts_work_start_time") else None)
            end = parse_time(row.get("ts_work_end_time").value if row.get("ts_work_end_time") else None)
            if not start or not end:
                continue
            raw = Decimal(str((datetime.combine(date.min, end) - datetime.combine(date.min, start)).seconds / 3600))
            breaks = number(row.get("ts_break_minutes").value if row.get("ts_break_minutes") else None)
            if breaks is None:
                breaks = number(self.c.value("ct_break_minutes")) or ZERO
            hours = max(ZERO, raw - breaks / 60)
            total += hours
            overtime += max(ZERO, hours - 8)
            days += 1
            daily.append({"date": work_date, "hours": hours})
        self.derived["ts_daily_actual_hours"] = daily
        self.derived["ts_period_actual_hours"] = total
        self.derived["ts_monthly_actual_hours"] = total
        return total, overtime, days

    def r02(self) -> RuleResult:
        record_available = bool_value(self.t.value("ts_record_available", True))
        missing_break = any(
            row.get("ts_break_minutes") is None or row["ts_break_minutes"].value is None
            for row in self.t.rows
        )
        actual, overtime, days = self._timesheet_hours()
        period_start = parse_date(self.p.value("ps_pay_period_start"))
        period_end = parse_date(self.p.value("ps_pay_period_end"))
        scheduled_days = None
        if period_start and period_end:
            scheduled_days = Decimal(sum(
                1 for offset in range((period_end - period_start).days + 1)
                if (period_start + timedelta(days=offset)).weekday() < 5
            ))
        paid_days = scheduled_days or number(self.p.value("ps_paid_work_days"))
        paid = sum((number(self.p.value(key)) or ZERO for key in
                    ("ps_paid_regular_hours", "ps_paid_overtime_hours", "ps_paid_holiday_hours")), ZERO)
        paid_overtime = number(self.p.value("ps_paid_overtime_hours")) or ZERO
        total_gap, overtime_gap = actual - paid, overtime - paid_overtime
        coverage = Decimal(days) / paid_days if paid_days else ZERO
        self.derived.update(cmp_actual_hours_gap=total_gap,
                            cmp_overtime_hours_gap=overtime_gap,
                            ts_period_coverage=coverage)
        if record_available is False or not self.t.rows:
            return self.result("R02", RuleStatus.NOT_CHECKABLE,
                               "근무기록이 없어 실근로시간을 평가할 수 없습니다.")
        if paid_days and coverage < self.parameters["param_timesheet_min_coverage"]:
            return self.result("R02", RuleStatus.NOT_CHECKABLE,
                               "근무기록 커버리지가 90% 미만입니다.", coverage=coverage)
        if missing_break:
            return self.result("R02", RuleStatus.REVIEW,
                               "근무기록의 휴게시간이 없어 계약값으로 대체했습니다.")
        total_tol = min(max(Decimal(days) * Decimal("0.083"), Decimal("0.5")),
                        paid * Decimal("0.03")) if paid else Decimal("0.5")
        overtime_tol = max(Decimal("0.5"), Decimal(sum(
            1 for item in self.derived["ts_daily_actual_hours"] if item["hours"] > 8
        )) * Decimal("0.083"))
        if abs(total_gap) > total_tol or abs(overtime_gap) > overtime_tol:
            return self.result("R02", RuleStatus.MISMATCH,
                               "근무기록과 명세서 계산시간이 허용범위를 벗어납니다.")
        return self.result("R02", RuleStatus.PASS,
                           "근무시간이 허용범위 내에서 일치합니다.")

    def r03(self) -> RuleResult:
        unmatched = [
            line.get("item_name") for line in (self.p.value("ps_pay_lines") or [])
            if line.get("mapped_to") is None and (number(line.get("amount")) or ZERO) != ZERO
        ]
        checks = []
        for contract_field, payslip_field, label in (
            ("ct_bonus_amount", "ps_bonus_amount", "상여금"),
            ("ct_extra_pay_amount", "ps_other_allowance", "기타수당"),
        ):
            expected, actual = number(self.c.value(contract_field)), number(self.p.value(payslip_field))
            if expected is not None:
                checks.append((label, actual or ZERO, expected, self.parameters["param_fixed_allowance_tolerance"]))
        overtime_hours = number(self.p.value("ps_paid_overtime_hours")) or ZERO
        overtime_rate = number(self.c.value("ct_overtime_hourly_pay"))
        if overtime_hours and overtime_rate is not None:
            checks.append(("연장수당", number(self.p.value("ps_overtime_pay")) or ZERO,
                           overtime_rate * overtime_hours,
                           self.parameters["param_rate_allowance_base_tolerance"]
                           + self.parameters["param_rate_allowance_hour_tolerance"] * overtime_hours))
        failed = [(label, actual - expected, tolerance)
                  for label, actual, expected, tolerance in checks
                  if abs(actual - expected) > tolerance]
        self.derived["cmp_unmatched_pay_items"] = unmatched
        if failed:
            return self.result("R03", RuleStatus.MISMATCH, "수당 지급액이 계약 기준과 다릅니다.", failed=failed)
        if unmatched:
            return self.result("R03", RuleStatus.REVIEW, "계약 근거가 없는 지급항목이 있습니다.", items=unmatched)
        if not checks:
            return self.result("R03", RuleStatus.NOT_CHECKABLE, "비교할 수당 조건이 없습니다.")
        return self.result("R03", RuleStatus.PASS, "수당과 상여금이 허용범위 내에서 일치합니다.")

    def r06(self) -> RuleResult:
        fields = ("ps_national_pension", "ps_health_insurance", "ps_long_term_care_insurance",
                  "ps_employment_insurance", "ps_income_tax", "ps_local_income_tax",
                  "ps_housing_deduction", "ps_meal_deduction", "ps_other_deduction")
        values = [number(self.p.value(key)) for key in fields]
        complete = all(value is not None for value in values)
        calc = sum((value or ZERO for value in values), ZERO)
        printed = number(self.p.value("ps_total_deduction"))
        self.derived.update(calc_total_deduction=calc, cond_deduction_components_complete=complete)
        if not complete or printed is None:
            return self.result("R06", RuleStatus.NOT_CHECKABLE, "공제항목이 누락되어 검산할 수 없습니다.")
        gap = calc - printed
        self.derived["cmp_total_deduction_gap"] = gap
        return self.result("R06", RuleStatus.PASS if gap == ZERO else RuleStatus.MISMATCH,
                           "공제항목 합계가 일치합니다." if gap == ZERO else "공제항목 합계가 다릅니다.")

    def r07(self) -> RuleResult:
        gross = self.derived.get("calc_gross_pay")
        deduction = self.derived.get("calc_total_deduction")
        printed = number(self.p.value("ps_net_pay"))
        if gross is None or deduction is None:
            return self.result("R07", RuleStatus.NOT_CHECKABLE, "총지급액 또는 총공제액이 없습니다.")
        calc = gross - deduction
        self.derived["calc_net_pay"] = calc
        if printed is None:
            return self.result("R07", RuleStatus.REVIEW, "실수령액 확인이 필요합니다.", calculated=calc)
        gap = calc - printed
        self.derived["cmp_net_pay_gap"] = gap
        if calc <= ZERO:
            return self.result("R07", RuleStatus.REVIEW_HIGH, "재계산 실수령액이 0 이하입니다.", gap=gap)
        return self.result("R07", RuleStatus.PASS if gap == ZERO else RuleStatus.MISMATCH,
                           "실수령액이 일치합니다." if gap == ZERO else "재계산 실수령액과 인쇄값이 다릅니다.", gap=gap)

    def _selected_transactions(self) -> list[dict[str, Any]]:
        employer = normalize_name(self.c.value("ct_employer_name"))
        selected = []
        for row in self.b.rows:
            values = {key: value.value for key, value in row.items()}
            explicit = bool_value(values.get("user_selected_salary_transaction"))
            text = " ".join(str(values.get(key) or "") for key in
                            ("bk_transaction_content", "bk_transfer_memo"))
            record = normalize_name(values.get("bk_transaction_record"))
            candidate = bool(employer and record == employer) or "급여" in text
            if explicit is True or (explicit is None and candidate):
                selected.append(values)
        return selected

    def r08(self) -> RuleResult:
        calculated_net = self.derived.get("calc_net_pay")
        printed_net = number(self.p.value("ps_net_pay"))
        selected = self._selected_transactions()
        total = sum((number(row.get("bk_deposit_amount")) or ZERO for row in selected), ZERO)
        self.derived.update(calc_salary_deposit_count=len(selected),
                            calc_salary_deposit_total=total,
                            selected_salary_transactions=selected)
        if calculated_net is None or not selected:
            return self.result("R08", RuleStatus.NOT_CHECKABLE, "확정된 급여 입금 거래가 없습니다.")
        targets = [calculated_net]
        if printed_net is not None:
            targets.append(printed_net)
        target = min(targets, key=lambda value: abs(total - value))
        gap = total - target
        self.derived["cmp_deposit_gap"] = gap
        self.derived["salary_payment_target"] = target
        return self.result("R08", RuleStatus.PASS if gap == ZERO else RuleStatus.MISMATCH,
                           "급여 입금합계가 실수령액과 일치합니다." if gap == ZERO else "급여 입금합계가 실수령액과 다릅니다.")

    def _scheduled_payment_date(self) -> date | None:
        period_end = parse_date(self.p.value("ps_pay_period_end"))
        if not period_end:
            return None
        timing = str(self.c.value("ct_pay_timing", "next_month")).lower()
        add_month = timing != "same_month"
        year, month = period_end.year, period_end.month + (1 if add_month else 0)
        if month == 13:
            year, month = year + 1, 1
        if str(self.c.value("ct_pay_cycle", "")).lower() == "weekly":
            weekday = str(self.c.value("ct_pay_weekday", "FRI")).upper()
            target = {"MON": 0, "TUE": 1, "WED": 2, "THU": 3, "FRI": 4, "SAT": 5, "SUN": 6}.get(weekday, 4)
            first = date(year, month, 1)
            return first + timedelta(days=(target - first.weekday()) % 7)
        day = number(self.c.value("ct_pay_day"))
        if day is None:
            return None
        return date(year, month, min(int(day), calendar.monthrange(year, month)[1]))

    def r09(self) -> RuleResult:
        net = self.derived.get("salary_payment_target", self.derived.get("calc_net_pay"))
        selected = self.derived.get("selected_salary_transactions", self._selected_transactions())
        scheduled = self._scheduled_payment_date()
        if net is None or not selected or scheduled is None or net <= ZERO:
            return self.result("R09", RuleStatus.NOT_CHECKABLE, "전액 지급 완료일을 확정할 수 없습니다.")
        cumulative, completion = ZERO, None
        for row in sorted(selected, key=lambda item: parse_datetime(item.get("bk_transaction_datetime")) or datetime.max):
            cumulative += number(row.get("bk_deposit_amount")) or ZERO
            if cumulative >= net:
                parsed = parse_datetime(row.get("bk_transaction_datetime"))
                completion = parsed.date() if parsed else None
                break
        self.derived.update(calc_scheduled_payment_date=scheduled,
                            calc_actual_payment_completion_date=completion)
        if completion is None:
            return self.result("R09", RuleStatus.NOT_CHECKABLE, "입금 누적액이 실수령액에 도달하지 않았습니다.")
        gap = (completion - scheduled).days
        return self.result("R09", RuleStatus.PASS if gap <= 0 else RuleStatus.MISMATCH,
                           "약정 지급일까지 전액 지급되었습니다." if gap <= 0 else "전액 지급 완료일이 약정 지급일보다 늦습니다.", gap_days=gap)

    def r10(self) -> RuleResult:
        actual = self.derived.get("ts_monthly_actual_hours")
        contract = number(self.c.value("ct_monthly_work_hours"))
        if not self.t.rows or actual is None or contract is None:
            return self.result("R10", RuleStatus.NOT_CHECKABLE, "계약시간 또는 근무기록이 없습니다.")
        period_start = parse_date(self.p.value("ps_pay_period_start"))
        period_end = parse_date(self.p.value("ps_pay_period_end"))
        if (normalize_wage_type(self.c.value("ct_wage_type")) == "monthly"
                and period_start and period_end
                and (period_start.day != 1
                     or period_end.day != calendar.monthrange(period_end.year, period_end.month)[1])):
            actual = contract
            self.derived["ts_monthly_actual_hours"] = actual
        gap = actual - contract
        daily = self.derived.get("ts_daily_actual_hours", [])
        max_daily = number(self.c.value("ct_max_daily_work_hours"))
        exceeded = [item for item in daily if max_daily is not None and item["hours"] > max_daily]
        confirmed = bool_value(self.t.value("ts_worker_confirmed")) is True
        self.derived.update(cmp_contract_actual_hours_gap=gap,
                            cond_daily_limit_exceeded=bool(exceeded),
                            cond_worker_confirmed=confirmed)
        if not confirmed or exceeded or abs(gap) > self.parameters["param_hours_tolerance"]:
            return self.result("R10", RuleStatus.REVIEW, "계약시간 차이 또는 일일 상한을 확인해야 합니다.", gap=gap)
        return self.result("R10", RuleStatus.PASS, "계약시간 대비 차이가 허용범위 이내입니다.", gap=gap)

    def r11(self) -> RuleResult:
        period_start = parse_date(self.p.value("ps_pay_period_start"))
        period_end = parse_date(self.p.value("ps_pay_period_end"))
        if period_start and period_end and period_start.year != period_end.year:
            return self.result("R11", RuleStatus.REVIEW,
                               "급여 산정기간이 두 연도에 걸쳐 있어 적용 최저임금 확인이 필요합니다.")
        minimum = number(self.parameters["param_minimum_hourly_wage"])
        contract, payslip = self.derived.get("calc_contract_hourly_wage"), self.derived.get("calc_payslip_hourly_wage")
        if minimum is None or (contract is None and payslip is None):
            return self.result("R11", RuleStatus.NOT_CHECKABLE, "최저임금 비교에 필요한 시급이 없습니다.")
        gaps = {"contract": contract - minimum if contract is not None else None, "payslip": payslip - minimum if payslip is not None else None}
        self.derived.update(cmp_contract_minimum_wage_gap=gaps["contract"], cmp_payslip_minimum_wage_gap=gaps["payslip"])
        if any(v is not None and v < 0 for v in gaps.values()):
            return self.result("R11", RuleStatus.REVIEW_HIGH, "2026년 기준시급 10,320원 미만 가능성이 있어 우선 확인이 필요합니다. 법 위반 확정 판정은 아닙니다.", **gaps)
        return self.result("R11", RuleStatus.PASS, "계약·명세서 시급이 최저임금 기준 이상입니다.")


## 5. 사용자 확인 게이트와 파이프라인


In [ ]:
from __future__ import annotations



CRITICAL_REVIEW_FIELDS = {
    "contract": {"ct_employee_name", "ct_new_or_reentry_start_date", "ct_new_or_reentry_end_date", "ct_pay_day"},
    "timesheet": {"ts_employee_name"},
    "payslip": {"ps_employee_name", "ps_pay_period_start", "ps_pay_period_end", "ps_gross_pay", "ps_total_deduction"},
}


class VitaminPipeline:
    def __init__(self, engine: RuleEngine | None = None, require_review: bool = True) -> None:
        self.engine = engine or RuleEngine()
        self.require_review = require_review

    def run(self, case: CaseBundle) -> EvaluationReport:
        pending = []
        if self.require_review:
            for doc_name, fields in CRITICAL_REVIEW_FIELDS.items():
                document = getattr(case, doc_name)
                for name in fields:
                    item = document.fields.get(name)
                    if item and not item.confirmed:
                        pending.append({"document": doc_name, "field": name, "value": item.value, "confidence": item.confidence})
        if pending:
            return EvaluationReport(case.case_id, {}, [], pending)
        derived, rules = self.engine.evaluate(case)
        return EvaluationReport(case.case_id, derived, rules)


## 6. JSON 입출력


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any



def load_case(path: str | Path) -> CaseBundle:
    path = Path(path)
    payload = json.loads(path.read_text(encoding="utf-8"))
    adapter = JsonOcrAdapter()
    docs = payload["documents"]
    # 실제 합성데이터는 JSON Schema v3.2의 canonical 값이므로 모두 확인 완료로 취급한다.
    canonical = (
        "bank_statement" in docs
        or "derived" in payload
        or "rule_results" in payload
    )
    return CaseBundle(
        contract=adapter.from_dict(docs["contract"], "contract", canonical=canonical),
        timesheet=adapter.from_dict(
            docs["timesheet"],
            "timesheet",
            row_key="lines" if "lines" in docs["timesheet"] else "rows",
            canonical=canonical,
        ),
        payslip=adapter.from_dict(docs["payslip"], "payslip", canonical=canonical),
        bank=adapter.from_dict(
            docs.get("bank_statement", docs.get("bank", {})),
            "bank",
            row_key="transactions" if "bank_statement" in docs else "rows",
            canonical=canonical,
        ),
        case_id=payload.get("case_id", path.stem),
    )


def write_json(path: str | Path, data: dict[str, Any]) -> None:
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


## 7. 사례 한 건 판정

`INPUT_CASE`만 새 OCR 추출 JSON 경로로 바꿔 실행하세요.


In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("실행결과")

def run_case(case_path, output_dir=OUTPUT_DIR):
    """OCR 추출 JSON 한 건을 판정하고 결과 JSON을 저장한다."""
    case_path, output_dir = Path(case_path), Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    source_data = json.loads(case_path.read_text(encoding="utf-8"))
    case = load_case(case_path)
    result = VitaminPipeline(require_review=True).run(case).to_dict()
    output_path = output_dir / f"{case.case_id}_report.json"

    # 합성데이터의 정답표는 판정 입력으로 쓰지 않고 사후 검증에만 쓴다.
    if "rule_results" in source_data:
        expected = {rule_id: value[f"{rule_id.lower()}_result"]
                    for rule_id, value in source_data["rule_results"].items()}
        actual = {item["rule_id"]: item["status"] for item in result["rules"]}
        result["validation"] = {
            "expected": expected,
            "matches": {rule_id: actual.get(rule_id) == status
                        for rule_id, status in expected.items()},
        }
    write_json(output_path, result)

    print("사례:", case.case_id)
    print("결과 파일:", output_path)
    for item in result["rules"]:
        print(f'{item["rule_id"]}: {item["status"]}')
    if result["review_required"]:
        print("사용자 확인 필요:", result["review_required"])
    return result

# 실제 사용할 때는 이 경로만 새 OCR 추출 JSON으로 바꾼다.
INPUT_CASE = Path("합성데이터/단일 오류/R02/CASE-R02-2026-01-0001/CASE-R02-2026-01-0001.json")
result = run_case(INPUT_CASE)


## 8. 선택 기능: 폴더 일괄 판정

운영의 기본 단위는 한 건이며, 이 셀은 회귀시험이나 대량 처리 때만 사용합니다.


In [ ]:
def run_folder(input_folder, output_dir=OUTPUT_DIR):
    """선택 기능: 폴더 안의 사례 JSON을 재귀적으로 일괄 판정한다."""
    files = sorted(Path(input_folder).rglob("*.json"))
    summaries = []
    for path in files:
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
            if "documents" not in payload:
                continue
            result = run_case(path, output_dir)
            summaries.append({"source": str(path), "case_id": result["case_id"],
                              "rules": result["rules"],
                              "review_required": result["review_required"]})
        except Exception as exc:
            summaries.append({"source": str(path), "error": str(exc)})
    write_json(Path(output_dir) / "batch_summary.json",
               {"total": len(summaries), "cases": summaries})
    return summaries

# 여러 건을 처리할 때만 아래 주석을 해제한다.
# batch_results = run_folder("입력_폴더_경로")
